**MDS7203 Modelos Generativos Profundos, Primavera 2023**

# Laboratorio 2: Modelo de lenguaje auto-regresivo

**Profesor**: Felipe Tobar, **Auxiliares**: Cristóbal Alcázar, Camilo Carvajal Reyes, **Ayudante**: Joaquín Barceló.

**Fecha de entrega**: viernes 29 de septiembre, 2023.

**Nombre: Fernando Fêtis Riquelme**

In [1]:
import torch
import torch.nn as nn
from torch import optim
import math
import plotly.graph_objects as go

torch.manual_seed(19_476_309_3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Se entrenó el mismo modelo sobre ambos corpus (Harry Potter y Shakespeare). Para cambiar entre un corpus y otro, modificar la constante `CORPUS` en la línea siguiente. En el corpus de Harry Potter se eliminaron un par de sinogramas (caracteres chinos) ya que correspondían a un link dentro del corpus y no se espera que formen parte del vocabulario. Todo el resto fue dejado igual. Además, el corpus de Harry Potter fue entrenado sobre una arquitectura más grande.

In [2]:
CORPUS = ('harry potter', 'shakespeare')[0]  # Harry Potter.
TRAIN = False  # se cargará el modelo entrenado.

## 1. Corpus

### <span style="color:orange">Lectura y vocabulario</span>

In [3]:
with open(f'data/{CORPUS}.txt', 'r', encoding='utf-8') as file:
    text = file.read()

print(f'Tamaño del corpus: {len(text):,} caracteres.')

Tamaño del corpus: 6,340,913 caracteres.


In [4]:
# Vocabulario:
vocabulary = sorted(list(set(text)))
vocab_size = len(vocabulary)

vocabulary_renamed = (
    ''.join(vocabulary)
    .replace('\t', '[TAB]')
    .replace('\n', '[NEWLINE]')
    .replace(' ', '[SPACE]')
)

print(f'Tamaño del vocabulario: {vocab_size} caracteres.')
print('Vocabulario (ordenado):', vocabulary_renamed)

Tamaño del vocabulario: 104 caracteres.
Vocabulario (ordenado): [TAB][NEWLINE][SPACE]!"$%&'()*,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz|}~é–—‘’“”…　


### <span style="color:orange">Funciones `encode` y `decode`</span>

><span style="color:yellow">Pregunta A:</span> Definir diccionarios para pasar de caracter a índice y viceversa. Con esto, definir las funciones `encode` y `decode` para tokenizar una secuencia.

In [5]:
char_to_index = {ch: i for i, ch in enumerate(vocabulary)}
index_to_char = vocabulary

# Funciones de encoding-decoding de tokens:
encode = lambda string: [char_to_index[ch] for ch in string]
decode = lambda tokens: ''.join(index_to_char[i] for i in tokens)

# Corpus tokenizado:
data = torch.tensor(encode(text), dtype=torch.long)

In [6]:
# Ejemplo de encoding y decoding:

string = 'abcde'
print(f'encode("{string}") = {encode(string)}.')
print(f'decode({encode(string)}) = "{decode(encode(string))}".')

encode("abcde") = [66, 67, 68, 69, 70].
decode([66, 67, 68, 69, 70]) = "abcde".


## 2. Procesamiento de datos

### <span style="color:orange">Hold-out</span>

Se entrenará con el 90% de los datos:

In [7]:
train_ratio = 0.9

n_train = int(0.9 * len(data))
train_data, eval_data = data[:n_train], data[n_train:]

print(f'Tamaño del corpus de entrenamiento: {len(train_data):>9,} caracteres.')
print(f'Tamaño del corpus de validación: {len(eval_data):>12,} caracteres.')

Tamaño del corpus de entrenamiento: 5,706,821 caracteres.
Tamaño del corpus de validación:      634,092 caracteres.


### <span style="color:orange">Bloques contextuales</span>

Surante el entrenamiento, dada una secuencia de tokens de largo `block_size`, se buscará predecir la palabra que se genera en cada tiempo, usando como entrada la subsecuencia hasta el tiempo anterior al que se quiere predecir.

Es decir, para una secuencia de entrenamiento $(x_1,\ldots, x_T)$ se utilizará como input a $(x_1,\ldots, x_t)$ para predecir $x_{t+1}$, para $t\in\{1,\ldots,T\}$ (es decir, se realizarán $T$ predicciones), por lo que se tendrá un vector target $(x_2,\ldots, x_{T+1})$.

In [8]:
# Ejemplo de bloque contextual:

block_size = 10

print(f'Ejemplo de bloque contextual (x, y):')
block_x = data[:block_size].tolist()
block_y = data[1:block_size+1].tolist()

print(f'x: {block_x} ("{decode(block_x)}")')
print(f'y: {block_y} ("{decode(block_y)}")')

Ejemplo de bloque contextual (x, y):
x: [41, 66, 83, 83, 90, 4, 49, 80, 85, 85] ("Harry Pott")
y: [66, 83, 83, 90, 4, 49, 80, 85, 85, 70] ("arry Potte")


In [9]:
# Pares (input, target) generados a partir de un bloque contextual de entrenamiento:

print('Pares de entrenamiento generados por el bloque anterior:\n')

print(f'{"Input":^64} Target')
for i in range(block_size):
    context = block_x[0:i+1]
    target = block_y[i]
    print(f'{str(context):<40} ({str(decode(context)) +")":<15} --> {target:5} ({decode([target])})')

Pares de entrenamiento generados por el bloque anterior:

                             Input                               Target
[41]                                     (H)              -->    66 (a)
[41, 66]                                 (Ha)             -->    83 (r)
[41, 66, 83]                             (Har)            -->    83 (r)
[41, 66, 83, 83]                         (Harr)           -->    90 (y)
[41, 66, 83, 83, 90]                     (Harry)          -->     4 ( )
[41, 66, 83, 83, 90, 4]                  (Harry )         -->    49 (P)
[41, 66, 83, 83, 90, 4, 49]              (Harry P)        -->    80 (o)
[41, 66, 83, 83, 90, 4, 49, 80]          (Harry Po)       -->    85 (t)
[41, 66, 83, 83, 90, 4, 49, 80, 85]      (Harry Pot)      -->    85 (t)
[41, 66, 83, 83, 90, 4, 49, 80, 85, 85]  (Harry Pott)     -->    70 (e)


Se creará una función que cree automáticamente los batches que se usarán durante el entrenamiento:

In [10]:
def get_batch(batch_size: int, block_size: int, from_train_set: bool = True, device: torch.device = device) -> tuple:
    '''
    Genera un batch de entrenamiento. Las secuencias serán construidas a partir de instantes aleatorios del corpus.

    Parameters:
        - batch_size: cantidad de secuencias del batch.
        - block_size: largo de cada secuencia.
        - from_train_set: indica si se obtendrán las secuencias a partir del train set.
        - device: dispositivo donde se almacenará el batch.

    Returns:
        - x_batch[batch_size, block_size]: secuencia continua para entrenamiento.
        - y_batch[batch_size, block_size]: secuencia de palabras que se deben generar.
    '''

    data = train_data if from_train_set else eval_data
    batch_indexes = torch.randint(len(data) - block_size - 1, (batch_size,))

    x_batch = torch.stack([data[i:i+block_size] for i in batch_indexes])
    y_batch = torch.stack([data[i+1:i+block_size+1] for i in batch_indexes])

    return x_batch.to(device), y_batch.to(device)

In [11]:
# Ejemplo de batch:

batch_size, block_size = 5, 10

print('Estructura de un batch:\n')
x_batch, y_batch = get_batch(batch_size, block_size)

assert x_batch.shape == y_batch.shape == torch.Size([batch_size, block_size])

for batch, batch_name in zip([x_batch, y_batch], ['x_batch', '\ny_batch']):
    print(f'{batch_name}:')
    for seq in batch.cpu().numpy():
        print(f'{str(seq):35} ("{decode(seq)}").')

Estructura de un batch:

x_batch:
[83 78 74 80 79 70 14  4 66 79]     ("rmione, an").
[70 78  4 73 70 83 84 70 77 71]     ("em herself").
[70 83  4 16 16 16  4 80 73  4]     ("er ... oh ").
[85 70 69  4 78 90  4 79 66 78]     ("ted my nam").
[42 71  4 66 79 90 67 80 69 90]     ("If anybody").

y_batch:
[78 74 80 79 70 14  4 66 79 69]     ("mione, and").
[78  4 73 70 83 84 70 77 71  4]     ("m herself ").
[83  4 16 16 16  4 80 73  4 68]     ("r ... oh c").
[70 69  4 78 90  4 79 66 78 70]     ("ed my name").
[71  4 66 79 90 67 80 69 90  4]     ("f anybody ").


## 3. Baseline

Se usará como modelo base el modelo propuesto en el enunciado. Este consite en utilizar el embedding del tiempo $t$ directamente como log-odds para la predicción de la palabra $t+1$, sin utilizar el resto de tiempos anteriores. Es decir, representa el modelo markoviano $p_\theta(y_{t+1}|x_1,\ldots,x_t) = p_\theta(y_{t+1}|x_t)$, donde $p_\theta(y_{t+1}|x_t)$ viene dado por $\text{softmax}(f_\theta(x_t))$, donde $f$ es una red neuronal.

In [12]:
class ARMBaseline(nn.Sequential):
    '''
    Modelo autorregresivo base. Calcula el logit (log-odds) de la predicción para cada instante de tiempo utilizando como contexto
    únicamente al tiempo actual. Todas las secuencias del batch deben ser del mismo tamaño.

    Forward parameters:
        - input[batch_size, seq_length]: batch de vectores de contexto
            (secuencias).

    Forward returns:
        - output[batch_size, seq_length, vocab_size]: predicciones (logits) para
        cada tiempo de cada secuencia.

    '''

    def __init__(self, vocab_size: int):
        '''
        Parameters:
            - vocab_size: tamaño del vocabulario.
        '''
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, vocab_size)

In [13]:
# Ejemplo de modelo base:

batch_size, seq_length = 3, 15

baseline = ARMBaseline(vocab_size)
input_baseline = torch.randint(vocab_size, size=[batch_size, seq_length])
output_baseline = baseline(input_baseline)

assert output_baseline.shape == torch.Size([batch_size, seq_length, vocab_size])

Para la generación de texto dado un contexto, se utilizará la función `generate_text` implementada más abajo.

## 4. Bloque decoder

### <span style="color:orange">Scaled dot-product attention</span>

En esta sección se responderán las dos preguntas asociadas a la forma de computar la matriz de atención.

><span style="color:yellow">Pregunta B:</span> Explicar por qué no se utilizan directamente los embeddings para computar la matriz de scores de atención (previo a la normalización) en vez de usar las proyecciones $K$ y $Q$.

Para una secuencia de largo $T$ y dimensión de cada token $d$, la matriz $X$ que almacena la secuencia es de tamaño $T\times d$ (considerando que cada fila es un token distinto). A partir de esto se construyen las matrices $Q,K\in\mathbb{R}^{T\times d_k}$ y $V\in\mathbb{R}^{T\times d_v}$ mediante una proyección lineal de la matriz $X$. De esta forma, la matriz de atención es calculada mediante

$$
\text{softmax}_\text{por filas}\left(\frac{QK^\top}{\sqrt{d_k}}\right)
$$

Hay dos motivos importantes para realizar una proyección de los datos antes de calcular los scores de atención:

- **Agregar más variabilidad:** Si se calculan los scores como $\frac{XX^\top}{\sqrt{d}}$ en vez de $\frac{QK^\top}{\sqrt{d_k}}$, entonces dos instantes de tiempo $t$ y $s$ tendrán una alta atención entre sí solo si sus representaciones (las filas $t$ y $s$ de $X$) son similares, ya que el producto punto mide similitud. Por lo tanto, permitiéndole al modelo aprender representaciones previas al cálculo de atención se podrá aprender otras similitudes que sean de más alto nivel (cada cabezal de atención tendrá un criterio de similitud diferente) y que sean más útiles para el modelo.
- **Evitar una matriz de atención de diagonal:** al calcular la matriz de scores mediante $\frac{XX^\top}{\sqrt{d}}$ se tendrá que el score de atención entre dos tiempos $t$ y $s$ será $\text{score}(s,t) = \frac{\langle x_s, x_t\rangle}{\sqrt{d}}$. Por lo tanto, el score será máximo cuando $s=t$ (de acuerdo a la desigualdad de Cauchy-Schwarz y asumiendo embeddings normalizados). Esto indica que al no proyectar la secuencia $X$ antes de calcular la atención provocará que siempre se le dé mayor atención al tiempo actual.

><span style="color:yellow">Pregunta C:</span> Explicar las razones que hay para escalar la matriz de scores de atención por $\frac{1}{\sqrt{d_k}}$, donde $d_k$ es la dimensión interna de los cabezales de atención.

Veremos que el escalamiento de la matriz de scores por $\sqrt{d_k}$ permite normalizar la varianza de cada uno de los scores.

Cada score de atención no normalizado (entrada de la matriz $QK^\top$) viene dado por el producto punto de una fila $q$ y una fila $k$. Asumiendo que los elementos de los vectores $q,k\in\mathbb{R}^{d_k}$ son independientes y provienen de una distribución normal $q_i, k_i\sim\mathcal{N}(0,1)$ (esto se logra mediante capas de normalización en la red neuronal), entonces:

$$
\langle q, k\rangle = \sum_{i=1}^{d_k} q_i k_i \implies \text{Var}(\langle q, k\rangle ) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = \sum_{i=1}^{d_k} \left(\mathbb{E}[(q_i k_i)^2] - \mathbb{E}^2[q_i k_i]\right)
$$


Por independencia, vemos que $\mathbb{E}[q_i k_i] = \mathbb{E}[q_i] \mathbb{E}[k_i]$ y que $\mathbb{E}[(q_i k_i)^2] = \mathbb{E}[q_i^2] \cdot \mathbb{E}[k_i^2]$, donde $\mathbb{E}[q_i] = 0$ y $\mathbb{E}[q_i^2] = \text{Var}(q_i) + \mathbb{E}^2[q_i]=1$ (lo mismo para $k_i$), luego:

$$
\text{Var}(\langle q, k\rangle) = \sum_{i=1}^{d_k} (1-0) = d_k
$$

Por lo tanto, $\text{score}(q,k) = \frac{\langle q,k\rangle}{\sqrt{d_k}}$ tendrá varianza unitaria. De esta forma, la normalización permite mantener (con alta probabilidad) la matriz de score dentro de un rango acotado, lo cual es deseable al momento de aplicar `softmax` ya que sin esto puede haber un problema de desbordamiento numérico.

### <span style="color:orange">Módulos de atención</span>

En esta sección se implementarán los dós módulos de atención usados en el decoder: `SelfAttentionHead` y `MultiHeadAttention`.

##### Cabezal de atención

En la implementación de los módulos asociados a un Transformer-decoder se permitirá variar todos los hiperparámetros de la arquitectura (dimensión de entrada, dimensión interna de los cabezales, dimensión de salida de los cabezales, cantidad de cabezales, etc.) en el constructor. Esto con el fin de dejar el código lo más modular posible y así poder adaptarlo fácilmente a otras arquitecturas que se proponen en la literatura. Los hiperparámetros finales serán indicados al momento de crear el modelo que se entrenará.

><span style="color:yellow">Pregunta D:</span> Implementar un cabezal de atención que utilice un módulo de self-attention (con máscara).

In [14]:
class SelfAttentionHead(nn.Module):
    '''Clase para un cabezal de autoatención.'''

    def __init__(self, input_dim: int, head_output_dim: int, head_dim: int, dropout: float):
        '''
        Parameters:
            - input_dim: dimensión de entrada al cabezal.
            - head_output_dim: dimensión de salida del cabezal (d_v).
            - head_dim: dimensión interna del cabezal (d_k).
            - dropout: tasa de dropout usado en los vectores de atención.
        '''
        super().__init__()

        self.query = nn.Linear(input_dim, head_dim)
        self.key = nn.Linear(input_dim, head_dim)
        self.value = nn.Linear(input_dim, head_output_dim)

        self.dropout = nn.Dropout(dropout)


    def forward(self, input: torch.Tensor) -> torch.Tensor:
        '''
        Calcula masked self-attention sobre el input utilizando un único cabezal.

        Parameters:
            - input[batch_size, seq_length, input_dim]: entrada al cabezal.

        Returns:
            - output[batch_size, seq_length, head_output_dim]: salida del cabezal.
        '''

        Q = self.query(input)
        K = self.key(input)
        V = self.value(input)

        batch_size, seq_length, head_dim = Q.shape
        self_device = next(self.parameters()).device
        mask = torch.tril(torch.ones(seq_length, seq_length)).to(self_device)

        attn = Q @ K.transpose(-1, -2) / math.sqrt(head_dim)
        attn = attn.masked_fill(mask == 0, -1e10)
        attn = self.dropout(attn.softmax(dim=-1))

        return attn @ V

In [15]:
# Ejemplo de cabezal de atención:

input_dim, head_output_dim, head_dim, dropout = 20, 10, 5, 0.1

head = SelfAttentionHead(input_dim, head_output_dim, head_dim, dropout)
input_head = torch.randn([batch_size, seq_length, input_dim])
output_head = head(input_head)

assert output_head.shape == torch.Size([batch_size, seq_length, head_output_dim])

##### Atención multicabezal (MHSA)

Para implementar el módulo multicabezal (MHSA) es útil notar que la dimensión de entrada y de salida deben ser iguales ya que se debe realizar una conexión residual. De esta forma, este tipo de bloque no necesitará recibir una dimensión de salida.

In [16]:
class MultiHeadAttention(nn.Module):
    '''Clase para un módulo multicabezal.'''

    def __init__(self, input_dim: int, num_heads: int, head_dim: int, head_output_dim: int, dropout: float):
        '''
        Parameters:
            - input_dim: dimensión de entrada y salida al módulo MHSA.
            - num_heads: cantidad de cabezales de atención.
            - head_dim: dimensión interna de cada cabezal (d_k).
            - head_output_dim: dimensión de salida de cada cabezal (d_v).
            - dropout: tasa de dropout usado en los vectores de atención y posterior a la proyección
                de los cabezales concatenados.
        '''
        super().__init__()

        self.heads = nn.ModuleList([
            SelfAttentionHead(input_dim, head_output_dim, head_dim, dropout)
            for _ in range(num_heads)
        ])
        self.projection = nn.Linear(head_output_dim * num_heads, input_dim)
        self.dropout = nn.Dropout(dropout)


    def forward(self, input: torch.Tensor) -> torch.Tensor:
        '''
        Calcula atención multicabezal sobre el input. Para esto, utiliza varias cabezas independientes,
        concatena sus salidas y luego las proyecta en un espacio de menor dimensión.

        Parameters:
            - input[batch_size, seq_length, input_dim]: batch de entrada.

        Returns:
            - output[batch_size, seq_length, input_dim]: batch de salida.
        '''

        output = torch.cat([head(input) for head in self.heads], dim=-1)
        output = self.dropout(self.projection(output))
        return output

In [17]:
# Ejemplo de bloque MHSA:

num_heads = 6

mhsa = MultiHeadAttention(input_dim, num_heads, head_dim, head_output_dim, dropout)
input_mhsa = torch.randn([batch_size, seq_length, input_dim])
output_mhsa = mhsa(input_mhsa)

assert output_mhsa.shape == torch.Size([batch_size, seq_length, input_dim])

### <span style="color:orange">Red feed-forward</span>

Al igual que para el módulo MHSA, la dimensión de entrada y salida del bloque feed forward debe ser la misma ya que la salida de la red feed forward será la entrada de otro bloque MHSA. Por lo tanto, en este módulo tampoco se necesitará indicar una dimensión de salida.

La red es aplicada independientemente en cada instante de tiempo de la secuencia (equivale a una convolución 1D).

><span style="color:yellow">Pregunta E:</span> Implementar la clase `FeedForward`.

In [18]:
class FeedForward(nn.Sequential):
    '''
    Aplica una red MLP de una capa oculta a un batch de secuencias de forma independiente
    en cada tiempo.

    Forward parameters:
        -input[batch_size, seq_length, input_dim].

    Forward returns:
        -output[batch_size, seq_length, input_dim].
    '''

    def __init__(self, input_dim: int, feedforward_factor: int, dropout: float):
        '''
        Parameters:
            - input_dim: dimensión de entrada y salida a la red.
            - feedforward_factor: factor de amplificación dentro de la red (dimensión oculta).
            - dropout: tasa de dropout usado en la salida.
        '''
        super().__init__()

        self.fc1 = nn.Linear(input_dim, input_dim * feedforward_factor)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(input_dim * feedforward_factor, input_dim)
        self.dropout = nn.Dropout(dropout)

In [19]:
# Ejemplo de red feed-forward:

feedforward_factor = 4

feedforward = FeedForward(input_dim, feedforward_factor, dropout)
input_feedforward = torch.randn([batch_size, seq_length, input_dim])
output_feedforward = feedforward(input_feedforward)

assert output_feedforward.shape == torch.Size([batch_size, seq_length, input_dim])

><span style="color:yellow">Pregunta F:</span> Explicar la relación entre los hiperparámetros que definen la cantidad de cabezales y la dimensión interna de cada cabezal.

El parámetro `num_heads` indica la cantidad de cabezales y el parámetro `head_output_dim` indica la dimensión de salida de cada cabezal. Por lo tanto, al concatenar todos los cabezales se obtiene un tensor de `num_heads` $\cdot$ `head_output_dim` características. Dado que este producto puede ser muy grande, esta concatenación se proyecta linealmente para recuperar la dimensión original. Esto permite disminuir la cantidad de neuronas de las redes feed-forward posteriores.

### <span style="color:orange">Capa del transformer decoder</span>

Para esta implementación se utilizará pre-LN architecture tal como se recomienda en el paper de [Xiong et al.](https://arxiv.org/abs/2002.04745).

><span style="color:yellow">Pregunta G:</span> Implementar la clase `DecoderBlock`.

In [20]:
class DecoderBlock(nn.Module):
    '''Clase para una capa del decoder.'''

    def __init__(self, input_dim: int, num_heads: int, head_dim: int, head_output_dim: int, dropout: float, feedforward_factor: int):
        '''
        Parameters:
            - input_dim, num_heads, head_dim, head_output_dim, feedforward_factor: igual que antes.
            - dropout: tasa de dropout aplicado en los submódulos y luego de cada capa FC.
        '''
        super().__init__()

        self.mhsa = MultiHeadAttention(input_dim, num_heads, head_dim, head_output_dim, dropout)
        self.feed_forward = FeedForward(input_dim, feedforward_factor, dropout)
        self.norm1 = nn.LayerNorm(input_dim)
        self.norm2 = nn.LayerNorm(input_dim)
        self.dropout = nn.Dropout(dropout)


    def forward(self, input: torch.Tensor) -> torch.Tensor:
        '''
        Aplica un bloque encoder (posterior al embedding inicial) al input.

        Parameters:
            -input[batch_size, seq_length, input_dim].

        Returns:
            -output[batch_size, seq_length, input_dim]
        '''

        input = self.dropout(input + self.mhsa(self.norm1(input)))
        input = self.dropout(input + self.feed_forward(self.norm2(input)))
        return input

In [21]:
# Ejemplo de bloque decoder:

decoderblock = DecoderBlock(input_dim, num_heads, head_dim, head_output_dim, dropout, feedforward_factor)
input_decoderblock = torch.randn([batch_size, seq_length, input_dim])
output_decoderblock = decoderblock(input_decoderblock)

assert output_decoderblock.shape == torch.Size([batch_size, seq_length, input_dim])

## 5. Modelo GPT y entrenamiento

### <span style="color:orange">Módulo `GPT`</span>

En esta sección se implementará el módulo `GPT` usando los módulos anteriores. Todos los hiperparámetros del modelo están disponibles como parámetros de la clase y los parámetros usados están indicados en la sección de entrenamiento.

><span style="color:yellow">Pregunta H:</span> Implementar la clase `GPT`.

In [22]:
class GPT(nn.Module):
    '''Clase para el modelo GPT.'''

    def __init__(
            self,
            vocab_size: int,
            embedding_dim: int,
            max_seq_length: int,
            num_layers: int,
            num_heads: int,
            head_dim: int,
            head_output_dim: int,
            dropout: float,
            feedforward_factor: int):
        '''
        Parameters:
            - vocab_size: tamaño del vocabulario.
            - embedding_dim: dimensión de embedding.
            - max_seq_length: largo máximo de entrada.
            - num_layers: número de capas del decoder.
            - num_heads, head_dim, head_output_dim, dropout, feedforward_factor: igual que antes.
        '''
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(max_seq_length, embedding_dim)
        self.blocks = nn.Sequential(
            *[DecoderBlock(
                embedding_dim,
                num_heads,
                head_dim,
                head_output_dim,
                dropout,
                feedforward_factor)
              for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(embedding_dim)
        self.fc_output = nn.Linear(embedding_dim, vocab_size)

        self.apply(self._init_weights)


    def forward(self, input: torch.Tensor) -> torch.Tensor:
        '''
        Aplica una red GPT a un batch de tokens. Devuelve una distribución
        no normalizada sobre el vocabulario para cada instante de la secuencia.

        Parameters:
            - input[batch_size, seq_length]: batch de secuencias de tokens.

        Returns:
            -output[batch_size, seq_length, vocab_size]: batch de secuencia de
                distribuciones no normalizada (logits).
        '''

        batch_size, seq_length = input.shape

        token_embedding = self.token_embedding(input)
        self_device = next(self.parameters()).device
        positional_embedding = self.position_embedding(torch.arange(seq_length, device=self_device))

        output = token_embedding + positional_embedding
        output = self.norm(self.blocks(output))
        output = self.fc_output(output)
        return output
    
    
    def _init_weights(self, module: nn.Module) -> None:
        '''
        Redefine los parámetros para las capas lineales y el embedding de GPT.

        Parameters:
            - module: módulo con parámetros.
        '''
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

In [23]:
# Ejemplo de modelo GPT:

embedding_dim, max_seq_length, num_layers = 64, 1000, 3

gpt = GPT(vocab_size, embedding_dim, max_seq_length, num_layers, num_heads, head_dim, head_output_dim, dropout, feedforward_factor)
input_gpt = torch.randint(vocab_size, size=[batch_size, seq_length])
output_gpt = gpt(input_gpt)

assert output_gpt.shape == torch.Size([batch_size, seq_length, vocab_size])

### <span style="color:orange">Generación autorregresiva</span>

Antes de entrenar el modelo, se creará una función auxiliar `generate_text` para generar texto dado un contexto. Esta función será usada para evaluar cualitativamente el modelo durante entrenamiento y luego para hacer inferencia. En esta función se da la opción de cambiar la temperatura (parámetro `temperature`), la cantidad de tokens usados para generar el siguiente token (parámetro `top_k`) y también se puede agregar una penalización por repetición de tokens (parámetro `repetition_penalty`).

In [24]:
def generate_text(
        model: nn.Module,
        context: str,
        new_tokens: int,
        temperature: float,
        top_k: int = int(vocab_size * 0.7),
        repetition_penalty: float = 1.2,
        print_text: bool = True
    ) -> torch.Tensor:
    '''
    Genera la continuación de una secuencia de texto.

    Parameters:
        - model: modelo para la generación.
        - context: contexto inicial.
        - new_tokens: cantidad de tokens para generar.
        - temperature: temperatura del sampling.
        - top_k: top tokens para considerar en el sampling.
        - repetition_penalty: penalización para tokens repetidos.
        - print_text: imprimir el texto a medida que se genera.
    '''

    if print_text:
        print(context, end='')

    context = torch.tensor(encode(context), dtype=torch.long, device=device).unsqueeze(0)
    model.eval().to(device)

    for _ in range(new_tokens):
        logits = model(context)
        last_logits = logits[:, -1] / temperature

        # Penalización por repetición:
        for idx in set(context[0].tolist()):
            last_logits[0, idx] /= repetition_penalty

        # Top-k sampling:
        top_logits, top_indices = torch.topk(last_logits, top_k)
        odds = top_logits.softmax(dim=-1)
        next_index = top_indices.gather(1, torch.multinomial(odds, num_samples=1))

        context = torch.cat((context, next_index), dim=1)
        if print_text:
            print(decode(next_index[0].tolist()), end='')

    return context

In [25]:
# Ejemplo de completación de texto con el modelo baseline:

context, new_tokens, temperature = 'initial context', 50, 0.7

text_generated = generate_text(baseline, context, new_tokens, temperature)

initial contextLo(k;’<}UFa$0–~w…biK	.240NnIM”oyl‘H>|KivEz
"QV4…i1

### <span style="color:orange">Bucle de entrenamiento</span>

><span style="color:yellow">Pregunta I:</span> Implementar el bucle de entrenamiento y entrenar el modelo.

Se construirá una función independiente que calcule la pérdida. Esto permite cambiar fácilmente la función objetivo:

In [26]:
def calc_loss(model: nn.Module, x_batch: torch.Tensor, y_batch: torch.Tensor, loss_function: nn.Module) -> torch.Tensor:
    '''
    Se calculará la función de pérdida para un batch de entrenamiento de la forma descrita al comienzo.

    Parameters:
        - model: modelo desde el que se calculará la loss.
        - x_batch[batch_size, block_size]: batch de secuencias de contexto.
        - y_batch[batch_size, block_size]: batch de targets.
        - loss_function: función usada para el cálculo de la loss.

    Returns:
        - loss: valor de la función de pérdida para el batch dado.
    '''

    output = model(x_batch)
    batch_size, block_size, vocab_size = output.shape
    logits = output.view(batch_size * block_size, vocab_size)
    targets = y_batch.view(batch_size * block_size)
    return loss_function(logits, targets)

Con esta función, se constuye en bucle de entrenamiento estándar. Se agregó la opción de incluir un scheduler ya que el cambio en el código es mínimo y puede contruibuir considerablemente al entrenamiento.

Durante la validación, se da la opción de generar texto usando como contexto a `context_eval` para evaluar cualitativamente el modelo durante el entrenamiento.

In [27]:
def train_model(
        model: nn.Module,
        optimizer: optim.Optimizer,
        scheduler: optim.lr_scheduler,
        n_iters: int,
        batch_size: int,
        block_size: int,
        filename: str,
        context_eval: str = None,
        eval_interval: int = 1000) -> None:

    '''
    Entrena un modelo genérico utilizando la metodología descrita al comienzo. Se guardará el mejor modelo
    y las estadísticas del entrenamiento.

    Parameters:
        - model: modelo que se entrenará.
        - optimizer: optimizador para el modelo.
        - scheduler: scheduler para la tasa de aprendizaje.
        - filename: archivo donde se guardará el entrenamiento.
        - n_iters: cantidad de batches de entrenamientos usados para entrenar.
        - batch_size: tamaño de cada batch.
        - block_size: largo de cada secuencia de entrenamiento.
        - context_eval: genera texto a partir de este contexto en cada iteración de validación.
        - eval_interval: periodo para la evaluación de la pérdida en el test set.
    '''

    model.to(device)
    model.train()
    loss_function = nn.CrossEntropyLoss()
    training_log = {
        'best': {'loss': float('-inf'), 'model': model.state_dict(), 'optimizer': optimizer.state_dict()},
        'loss': {'train': [], 'eval': []}
    }
    if context_eval is not None:
        print(f'Contexto para validación: "{context_eval}".\n')

    try:
        for iter in range(1, n_iters + 1):

            x_batch, y_batch = get_batch(batch_size, block_size)
            loss = calc_loss(model, x_batch, y_batch, loss_function)
            training_log['loss']['train'].append(loss.item())

            # Entrenamiento:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            # Actualización del mejor modelo:
            if loss.item() < training_log['best']['loss']:
                training_log['best']['loss'] = loss.item()
                training_log['best']['model'] = model.state_dict().copy()
                training_log['best']['optimizer'] = optimizer.state_dict().copy()

            # Evaluación y respaldo:
            if iter == 1 or iter % eval_interval == 0:
                model.eval()
                with torch.no_grad():
                    x_batch, y_batch = get_batch(batch_size, block_size, from_train_set=False)
                    loss = calc_loss(model, x_batch, y_batch, loss_function)
                    training_log['loss']['eval'].append(loss.item())

                print(f'\nIteración {iter:<5} (loss: {loss:.4f})')
                torch.save(training_log, filename)

                if context_eval is not None:
                    print('-' * 100)
                    _ = generate_text(baseline, context_eval, new_tokens, temperature)
                    print('\n', '-' * 100)
                model.train()

    except KeyboardInterrupt:
        print('Entrenamiento interrumpido.')

    torch.save(training_log, filename)

Se implementará una segunda función auxiliar para mostrar la evolución de la función de pérdida durante el entrenamiento:

In [28]:
def plot_loss(loss_log: dict, eval_interval=1000) -> None:
    '''
    Muestra el gráfico de la evolución de la función de pérdida durante el entrenamiento.

    Parameters:
        - loss_log: diccionario con llaves 'train' y 'eval' que indican listas numéricas.
        - eval_interval: periodo de evaluación en el conjunto de validación (usado como step
            horizontal).
    '''

    fig = go.Figure()
    for loss in ('train', 'eval'):
        x_eval = torch.arange(0, len(loss_log['train']) + 1, eval_interval)
        y = loss_log[loss]
        fig.add_trace(go.Scatter(x=(x_eval if loss == 'eval' else None), y=y, name=loss))

    min_loss = min(loss_log['train'])
    idx_min_loss = loss_log['train'].index(min_loss)

    fig.add_vline(x=idx_min_loss, line_width=3, line_dash='dash', annotation_text='mejor iteración')
    fig.update_layout(
        title=f'Evolución de la función de pérdida para {CORPUS.capitalize()}',
        xaxis_title='Iteración',
        yaxis_title='Loss'
    )
    fig.show()

### <span style="color:orange">Entrenamiento del modelo GPT</span>

En esta sección se definirán los hiperparámetros que se utilizarán, considerando que por lo general, `head_dim` $*$ `num_heads` $=$ `embedding_dim` y que `head_output_dim` $=$ `head_dim`. Además, se utilizaron arquitecturas distintas dependiendo del corpus de entrenamiento. El modelo usado para Harry Potter es el más grande (3.3M de parámetros).

In [29]:
if CORPUS == 'harry potter':
    architecture_hp = {
        'vocab_size': vocab_size,
        'embedding_dim': 256,
        'max_seq_length': 512,
        'num_layers': 4,
        'num_heads': 4,
        'head_dim': 64,
        'head_output_dim': 64,
        'dropout': 0.2,
        'feedforward_factor': 4
    }
else:
    architecture_hp = {
        'vocab_size': vocab_size,
        'embedding_dim': 192,
        'max_seq_length': 256,
        'num_layers': 6,
        'num_heads': 6,
        'head_dim': 32,
        'head_output_dim': 32,
        'dropout': 0.1,
        'feedforward_factor': 4
    }

Para el entrenamiento se utilizarán los siguientes parámetros:

In [30]:
n_iters = (20000 if CORPUS == 'harry potter' else 10000)
batch_size = (150 if CORPUS == 'harry potter' else 256)
block_size = architecture_hp['max_seq_length']
context_eval = decode(eval_data[0:10])

Ambos corpus fueron entrenados utilizando `Adam`. También se agregó un scheduler para la tasa de aprendizaje (no se usó el mismo que se sugiere en el paper original ya que los modelos son de escalas distintas). Los hiperparámetros de entrenamiento usandos para cada corpus están definidos en el siguiente bloque:

In [31]:
gpt = GPT(**architecture_hp)
print(f'Cantidad de parámetros: {sum(p.numel() for p in gpt.parameters()):,}.')

Cantidad de parámetros: 3,343,976.


In [32]:
gpt_filename = f'training/{CORPUS} (gpt).pt'

if TRAIN:
    gpt_optimizer = optim.Adam(gpt.parameters(), lr=3e-4)
    gpt_scheduler = optim.lr_scheduler.OneCycleLR(gpt_optimizer, max_lr=3e-4, total_steps=20000, anneal_strategy='linear')
    train_model(gpt, gpt_optimizer, gpt_scheduler, n_iters, batch_size, block_size, gpt_filename, context_eval)

In [33]:
# Cargar mejor modelo y graficar la evolución del entrenamiento:

gpt_training_log = torch.load(gpt_filename, map_location=device)
gpt.load_state_dict(gpt_training_log['best']['model'])
plot_loss(gpt_training_log['loss'])

En el gráfico vemos que en las primeras iteraciones hay una caída considerable de la función de pérdida. Esto se debe a que el modelo rápidamente puede aprender patrones básicos como el uso mayoritario de las letras sobre los números, el uso de espacios y la construcción de digrafos frecuentes del inglés como `ch`, `th` y `wh`. En las siguientes iteraciones el modelo debe lograr aprender la semántica del corpus, lo cual es un trabajo más difícil, por lo que la evolución de la función de pérdida es más suave.

Por otro lado, se está eligiendo el mejor modelo según la evolución de la loss en el entrenamiento. Esto es debido a que el modelo se está entrenando de forma autosupervisada, por lo que no se espera que haya overfitting (asumiendo que el corpus es suficientemente grande). Por otro lado, es importante considerar que la loss de evaluación está siendo calculada con el modelo en modo `eval`, por lo que se espera que tenga mejor rendimiento que la evaluación realizada durante el entrenamiento.

## 6. Generación de secuencias

Para la generación de texto dado un contexto, se usará la función `generate_text` implementada más arriba.

In [34]:
def generate_samples(model: nn.Module, context: str, n_samples: int, temperature):
    print('CONTEXTO', '-' * 200, context, '-' * 200, sep='\n')
    new_tokens = architecture_hp['max_seq_length'] - len(context)  # máximo de tokens posibles.
    
    print(f'\nSECUENCIAS GENERADAS (temperature={temperature})')
    for _ in range(n_samples):
        print('-' * 200)
        _ = generate_text(model, context, new_tokens, temperature)
        print()
    print('-' * 200)

De esta forma, generar texto dado un contexto se puede hacer fácilmente:

In [35]:
prompt = 'Felipe Tobar, the new Defence Against the Dark Arts Professor'
generate_samples(gpt, prompt, n_samples=3, temperature=0.7)

CONTEXTO
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Felipe Tobar, the new Defence Against the Dark Arts Professor
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

SECUENCIAS GENERADAS (temperature=0.7)
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Felipe Tobar, the new Defence Against the Dark Arts Professor McGonagall's family.
　　"I've been an expection?" Harry asked at them. "Not this was pretending he's weak!"
　　"What?"
　　"You have you known you don't taken someone," said Ron. "Like you."
　　"Oh, there was why I don't know tha

Para el corpus de Shakespeare se observa que el modelo aprendió a generar texto en forma de guión. En particular, genera nombres de personajes de las obras, respeta las mayúsculas y la puntuación en general.

Para el corpus de Harry Potter (más grande que el corpus de Shakespeare) se observa que la generación del texto es en estilo narrativo. Además, el texto generado tiene más cohesión entre las palabras y respeta varias reglas gramaticales básicas del inglés. Se puede observar que el parámetro de temperatura puede cambiar significativamente la salida del modelo.

## 7. Modelo base

><span style="color:yellow">Pregunta J:</span> Entrenar el modelo baseline y comparar con el modelo GPT.

### <span style="color:orange">Entrenamiento del modelo base</span>

En esta sección se entrenará el modelo base `ARMBaseline` utilizando los mismos hiperparámetros de entrenamiento que los que se usaron en el entrenamiento de GPT. Si bien se puede aumentar el parámetro `batch_size`, se dejará igual para que la comparación sea justa.

In [36]:
baseline = ARMBaseline(vocab_size)
print(f'Cantidad de parámetros: {sum(p.numel() for p in baseline.parameters()):,}.')

Cantidad de parámetros: 10,816.


In [37]:
baseline_filename = f'training/{CORPUS} (baseline).pt'

if TRAIN:
    baseline_optimizer = optim.Adam(baseline.parameters(), lr=3e-4)
    baseline_scheduler = optim.lr_scheduler.OneCycleLR(baseline_optimizer, max_lr=3e-4, total_steps=20000, anneal_strategy='linear')
    train_model(baseline, baseline_optimizer, baseline_scheduler, n_iters, batch_size, block_size, baseline_filename)

In [38]:
# Cargar mejor modelo y graficar la evolución del entrenamiento:

baseline_training_log = torch.load(baseline_filename, map_location=device)
baseline.load_state_dict(baseline_training_log['best']['model'])
plot_loss(baseline_training_log['loss'])

### <span style="color:orange">Comparación con GPT</span>

Se mostrarán las curvas de entrenamiento de ambos modelos para comparar su evolución.

In [39]:
fig = go.Figure()

fig.add_trace(go.Scatter(y=gpt_training_log['loss']['train'], name='GPT'))
fig.add_trace(go.Scatter(y=baseline_training_log['loss']['train'], name='baseline'))

fig.update_layout(
    title=f'Comparación del entrenamiento del modelo GPT y el modelo base',
    xaxis_title='Iteración',
    yaxis_title='Loss'
)
fig.show()

En ambos corpus (Harry Potter y Shakespeare) se observa que el modelo GPT tiene mejor rendimiento. Además, se ve como la evolución del modelo base no es capaz de alcanzar la pérdida alcanzada en la caída inicial del modelo GPT, que es donde el modelo aprende patrones básicos del lenguaje.

Por otra parte, el modelo base es solamente un embedding, por lo que es equivalente a un modelo lineal. Por lo tanto, este modelo aprende mucho más lento a simular el texto en inglés. Se observa que la loss del modelo base puede seguir disminuyendo fácilmente, aunque es esperable que no pueda disminuir mucho más debido a la baja complejidad del modelo.

### <span style="color:orange">Generación de secuencias con el modelo base</span>

Se generará texto utilizando el mismo contexto que se usó para el modelo GPT:

In [40]:
generate_samples(baseline, prompt, n_samples=3, temperature=0.7)

CONTEXTO
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Felipe Tobar, the new Defence Against the Dark Arts Professor
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

SECUENCIAS GENERADAS (temperature=0.7)
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Felipe Tobar, the new Defence Against the Dark Arts Professorkid.. cr RoRoéeid harmblseritLEy as t H=- te’m in hite \9*
　"
　"Sirdl,cily aluNe owe ved,
Ho w. Has gr gheas ll" rche an tole t g[6; theyof…apARou Elois t t <Mard heviere' s, s sas atot t owhHall,Panang cl ge haze'lllybed ho

Aquí se puede ver que el modelo no fue capaz de aprender practicamente nada acerca del lenguage natural (para ninguno de los dos corpus). Lo más destacable podría ser el hecho de que aprendió que se usan más letras que símbolos y números, y que las palabras se separan por espacios, pero esto también podría ser resultado de la aleatoriedad.